In [1]:
# ============================================================
# Zero-shot NLI Baseline for Hallucination Detection
# Same data + same split as fine-tuned DeBERTa
# No training
# ============================================================

!pip -q install transformers accelerate sentencepiece pandas scikit-learn tqdm

import os, json, random
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ============================================================
# CONFIG
# ============================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_PATH = "/kaggle/input/datasets/belalheshamfathy/orrrrrrrr/qa_data.json"  # عدّل path لو مختلف

OUTPUT_DIR = "/kaggle/working/zeroshot_nli_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_NAME = "cross-encoder/nli-deberta-v3-base"
BATCH_SIZE = 8
MAX_LENGTH = 512

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# ============================================================
# LOAD JSON / JSONL
# ============================================================

def load_json_or_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        first_char = f.read(1)
        f.seek(0)

        if first_char == "[":
            data = json.load(f)
        else:
            for line in f:
                line = line.strip()
                if line:
                    data.append(json.loads(line))
    return data

raw = load_json_or_jsonl(DATA_PATH)

if isinstance(raw, dict):
    for k in ["data", "items", "records"]:
        if k in raw and isinstance(raw[k], list):
            raw = raw[k]
            break

df = pd.DataFrame(raw)

print("Original records:", len(df))
print("Columns:", df.columns.tolist())
print(df.head(1))

# ============================================================
# BUILD SAME CLASSIFICATION DATASET
# Each record -> 2 samples
# right_answer = supported = 1
# hallucinated_answer = hallucination = 0
# ============================================================

required_cols = ["knowledge", "question", "right_answer", "hallucinated_answer"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

rows = []

for _, r in df.iterrows():
    rows.append({
        "knowledge": r["knowledge"],
        "question": r["question"],
        "answer": r["right_answer"],
        "label": 1
    })
    rows.append({
        "knowledge": r["knowledge"],
        "question": r["question"],
        "answer": r["hallucinated_answer"],
        "label": 0
    })

clf_df = pd.DataFrame(rows)
clf_df = clf_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print("\nClassification samples:", len(clf_df))
print(clf_df["label"].value_counts())

# ============================================================
# SAME TRAIN / TEST SPLIT AS FINE-TUNED CODE
# We only evaluate zero-shot on test_df
# ============================================================

train_df, test_df = train_test_split(
    clf_df,
    test_size=0.30,
    random_state=SEED,
    stratify=clf_df["label"]
)

test_df = test_df.reset_index(drop=True)

print("\nTrain size:", len(train_df))
print("Test size:", len(test_df))
print("Test label distribution:")
print(test_df["label"].value_counts())

# Optional: save the exact test split
test_df.to_csv(os.path.join(OUTPUT_DIR, "same_test_split_6000.csv"), index=False)

# ============================================================
# LOAD ZERO-SHOT NLI MODEL
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(device)
model.eval()

print("\nModel labels:")
print(model.config.id2label)

# ============================================================
# NLI INPUT FORMAT
# Premise = Knowledge + Question
# Hypothesis = Answer
# entailment => supported = 1
# contradiction / neutral => hallucination = 0
# ============================================================

def build_premise(row):
    return (
        f"Knowledge: {row['knowledge']}\n"
        f"Question: {row['question']}"
    )

def build_hypothesis(row):
    return f"Answer: {row['answer']}"

def get_entailment_index(model):
    id2label = model.config.id2label
    for idx, label in id2label.items():
        if "entail" in label.lower():
            return int(idx)
    raise ValueError(f"Could not find entailment label in id2label: {id2label}")

entailment_idx = get_entailment_index(model)
print("Entailment index:", entailment_idx)

# ============================================================
# PREDICT
# ============================================================

all_preds = []
all_probs = []

with torch.inference_mode():
    for start in tqdm(range(0, len(test_df), BATCH_SIZE)):
        batch = test_df.iloc[start:start+BATCH_SIZE]

        premises = [build_premise(row) for _, row in batch.iterrows()]
        hypotheses = [build_hypothesis(row) for _, row in batch.iterrows()]

        inputs = tokenizer(
            premises,
            hypotheses,
            truncation=True,
            padding=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)

        entail_probs = probs[:, entailment_idx].detach().cpu().numpy()

        # Default threshold: entailment probability >= 0.5 means supported
        preds = (entail_probs >= 0.5).astype(int)

        all_probs.extend(entail_probs.tolist())
        all_preds.extend(preds.tolist())

test_labels = test_df["label"].astype(int).values
test_preds = np.array(all_preds)

# ============================================================
# METRICS
# ============================================================

acc = accuracy_score(test_labels, test_preds)
precision, recall, f1, _ = precision_recall_fscore_support(
    test_labels,
    test_preds,
    average="binary",
    pos_label=1,
    zero_division=0
)

print("\n=== ZERO-SHOT NLI TEST METRICS ===")
print("Accuracy :", acc)
print("Precision:", precision)
print("Recall   :", recall)
print("F1       :", f1)

print("\n=== Classification Report ===")
print(classification_report(
    test_labels,
    test_preds,
    target_names=["hallucination(0)", "supported(1)"],
    zero_division=0
))

print("\n=== Confusion Matrix ===")
cm = confusion_matrix(test_labels, test_preds)
print(cm)

# ============================================================
# SAVE OUTPUTS
# ============================================================

metrics_df = pd.DataFrame([{
    "method": "Zero-shot NLI DeBERTa",
    "model": MODEL_NAME,
    "accuracy": acc,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "test_samples": len(test_df),
    "threshold": 0.5
}])

metrics_df.to_csv(os.path.join(OUTPUT_DIR, "zeroshot_nli_metrics.csv"), index=False)

cm_df = pd.DataFrame(
    cm,
    columns=["Pred_Hallucination", "Pred_Supported"],
    index=["True_Hallucination", "True_Supported"]
)
cm_df.to_csv(os.path.join(OUTPUT_DIR, "zeroshot_nli_confusion_matrix.csv"))

out_df = test_df.copy()
out_df["entailment_probability"] = all_probs
out_df["prediction"] = test_preds
out_df["correct"] = out_df["prediction"] == out_df["label"]
out_df.to_csv(os.path.join(OUTPUT_DIR, "zeroshot_nli_predictions.csv"), index=False)

print("\nSaved outputs to:", OUTPUT_DIR)
print("- zeroshot_nli_metrics.csv")
print("- zeroshot_nli_confusion_matrix.csv")
print("- zeroshot_nli_predictions.csv")
print("- same_test_split_6000.csv")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 92.3 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 whic

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Model labels:
{0: 'contradiction', 1: 'entailment', 2: 'neutral'}
Entailment index: 1


  0%|          | 0/750 [00:00<?, ?it/s]


=== ZERO-SHOT NLI TEST METRICS ===
Accuracy : 0.528
Precision: 0.6042183622828784
Recall   : 0.16233333333333333
F1       : 0.255911718339464

=== Classification Report ===
                  precision    recall  f1-score   support

hallucination(0)       0.52      0.89      0.65      3000
    supported(1)       0.60      0.16      0.26      3000

        accuracy                           0.53      6000
       macro avg       0.56      0.53      0.46      6000
    weighted avg       0.56      0.53      0.46      6000


=== Confusion Matrix ===
[[2681  319]
 [2513  487]]

Saved outputs to: /kaggle/working/zeroshot_nli_outputs
- zeroshot_nli_metrics.csv
- zeroshot_nli_confusion_matrix.csv
- zeroshot_nli_predictions.csv
- same_test_split_6000.csv
